# _lib/type_helpers — Silver type-fix helpers

Pure helper functions for the Bronze→Silver type casts.
Reusable for `order_detail` and any future Silver tables.

**Import via `%run ./_lib/type_helpers` from the Silver DLT pipeline notebook.**

| Bronze column | Bronze type | Silver type |
|---|---|---|
| `SERVED_TS` | `StringType` | `TimestampType` |
| `ORDER_TAX_AMOUNT` | `StringType` | `DecimalType(38, 4)` |
| `ORDER_DISCOUNT_AMOUNT` | `StringType` | `DecimalType(38, 4)` |
| `ORDER_ITEM_DISCOUNT_AMOUNT` | `StringType` | `DecimalType(38, 4)` |
| `LOCATION_ID` | `DoubleType` | `DecimalType(38, 0)` |
| `DISCOUNT_ID` | `StringType` | `DecimalType(38, 0)` nullable |
| `SHIFT_START_TIME` | `IntegerType` (millis) | `StringType` `'HH:mm:ss'` |
| `SHIFT_END_TIME` | `IntegerType` (millis) | `StringType` `'HH:mm:ss'` |

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Column


def millis_to_hhmmss(col: Column) -> Column:
    """Convert an IntegerType column of milliseconds-since-midnight to
    a 'HH:mm:ss' StringType column.

    Bronze stores SHIFT_START_TIME and SHIFT_END_TIME as raw INT32
    TIME(MILLIS) values.  Silver wants human-readable 'HH:mm:ss' strings.

    Example: 36000000 ms  →  '10:00:00'
    """
    total_seconds = (col / 1000).cast("long")
    hh = F.lpad((total_seconds / 3600).cast("int").cast("string"), 2, "0")
    mm = F.lpad(((total_seconds % 3600) / 60).cast("int").cast("string"), 2, "0")
    ss = F.lpad((total_seconds % 60).cast("int").cast("string"), 2, "0")
    return F.concat(hh, F.lit(":"), mm, F.lit(":"), ss)


def cast_served_ts(col: Column) -> Column:
    """Parse SERVED_TS (StringType 'yyyy-MM-dd HH:mm:ss') to TimestampType."""
    return F.to_timestamp(col, "yyyy-MM-dd HH:mm:ss")


def cast_decimal_38_4(col: Column) -> Column:
    """Cast a StringType monetary column to DecimalType(38, 4)."""
    return col.cast("decimal(38, 4)")


def cast_decimal_38_0(col: Column) -> Column:
    """Cast a DoubleType or StringType ID column to DecimalType(38, 0).

    Used for LOCATION_ID (Double→Decimal) and DISCOUNT_ID (String→Decimal nullable).
    Returns NULL for NULL inputs (nullable semantics preserved).
    """
    return col.cast("decimal(38, 0)")


print("type_helpers loaded: millis_to_hhmmss, cast_served_ts, cast_decimal_38_4, cast_decimal_38_0")